<a href="https://colab.research.google.com/github/Agoston03/Machine-Learning-VIMIMA05/blob/main/ML_2026_lab7_BN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal discovery in probabilistic graphical models

Bayesian-networks simplify the visualization of conditional independence between variables and inference, and are therefore well suited for modelling different domains. Their structure (the existence and direction of graph edges) can be determined by experts, but it is common to want to retrieve the DAG structure that generated the data from data under given assumptions, the latter is called structure learning or causal discovery.

For this exercise we will use the [Sachs dataset](http://www.esalq.usp.br/lepse/imgs/conteudo_thumb/Causal-Protein-Signaling-Networks-Derived-from-Multiparameter-Single-Cell-Data.pdf). It contains signals between proteins with experimentally confirmed reference structures. The measurements will include cytometric data on the relative appearance of different molecules under different perturbations.

First, let's run the 5 cells below until the graph is loaded!

## Setup

In [ ]:
! pip install pgmpy --quiet
! pip install cdt --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.linalg as slin
import scipy.optimize as sopt
from scipy.special import expit as sigmoid
from sklearn.metrics import mutual_info_score

import pgmpy
from pgmpy.estimators import BIC
from cdt.data import load_dataset
import networkx as nx

np.random.seed(123)

In [ ]:
s_data, s_graph = load_dataset('sachs')
s_graph.remove_edge('PIP2', 'PIP3')
s_graph.add_edge('PIP3', 'PIP2')

In [ ]:
N_BINS = 4
discrete_s_data = s_data.copy()
for c in s_data.columns:
  discrete_s_data[c] = pd.cut(x=discrete_s_data[c], bins=[np.nanquantile(discrete_s_data[c], (1/N_BINS)*i)  if i!=0 else 0 for i in range(N_BINS+1)],
                     labels=[0,1,2,3])

## Exercise 0

Let's take a look at the graph below!

Optional: try to justify 1-2 interactions with sources! What is the role of the proteins under investigation? Are there any interactions that could not be confirmed with a few google searches?

In [ ]:
nx.draw(s_graph, pos=nx.circular_layout(s_graph), with_labels=True)
plt.title('Ground truth')
plt.show()

## Exercise 1: Mutual information-based associations



Let's construct a baseline method, with which we cannot obtain a DAG, but we can approximate the real graph with an undirected graph structure. To do this, compute the pairwise mutual information for each variable in a matrix (`mutual_info_score`)!  Then choose a threshold value that, when cut off, gives a "plausible" association graph! Plot and compare the resulting structure with the second cell! How does it differ from the original?

In [ ]:
data = discrete_s_data.to_numpy()
mut_infos = np.zeros((data.shape[1],data.shape[1]))

for i in range(data.shape[1]):
  for j in range(data.shape[1]):
    if(i!=j):
      mut_infos[i,j] = mutual_info_score(data[:,i], data[:,j])

threshold = 0.05

mut_infos[mut_infos < threshold] = 0

In [ ]:
nx_graph_mut_info = nx.from_numpy_array(mut_infos)
nx_graph_mut_info = nx.relabel_nodes(nx_graph_mut_info, {a:b for (a,b) in zip(range(data.shape[1]), s_data.columns)})
nx.draw(nx_graph_mut_info, pos=nx.circular_layout(s_graph), with_labels=True)
plt.title('Mutual information based associations')
plt.show()

TODO: Evaluate graph

## Exercise 2: Score-based discrete search




The idea of discrete search is to move between DAG states in such a way that they match our data as closely as possible. There are several search methods available, the one of choice now is Probabilistic Hill Climb. To do this, we generate the possible neighbours at each step, then calculate the score difference between them and select the next state based on a proportional probability.

### 2.1 Enumeration of neighbouring DAGs

We now consider two DAG states to be neighbours if they differ by only one edge (addition, deletion or flip). First, let's generate the possible neighbours! For this, we have written the functions for edge addition and deletion in advance, now let's add the reversal! As with the first two functions, the ```potential_edges_to_flip`` function should return a dicitionary with the key being the edge to flip and the value being the difference of the scores (be careful, here the parents of two nodes are changing).

In [ ]:
def potential_edges_to_add(model, data, score):
  pot_edges = {}
  for u in model.nodes():
    for v in model.nodes():
      if((u,v) not in model.edges() and not nx.has_path(model, v, u)):
        old_parents = model.get_parents(v)
        new_parents = model.get_parents(v)+[u]
        pot_edges[(u,v)] = score(v, new_parents) - score(v, old_parents)
  return pot_edges

def potential_edges_to_remove(model, data, score):
  pot_edges = {}
  for u,v in model.edges():
    old_parents = model.get_parents(v)
    new_parents = [p for p in model.get_parents(v) if p != u]
    pot_edges[(u,v)] = score(v, new_parents) - score(v, old_parents)
  return pot_edges

def potential_edges_to_flip(model, data, score):
  pot_edges = {}
  for u,v in model.edges():
    if(not np.any([nx.has_path(model, u, p) for p in model.get_parents(v) if p != u])):
      #TODO

      pot_edges[(u, v)] = ...#new score - old score
  return pot_edges


def generate_potential_actions(model, data, score):
  pot_add = potential_edges_to_add(model, data, score)
  pot_rem = potential_edges_to_remove(model, data, score)
  pot_flip = potential_edges_to_flip(model, data, score)

  pot_actions = []

  for ka in pot_add.keys():
    pot_actions.append(('add', ka, pot_add[ka]))
  for kr in pot_rem.keys():
    pot_actions.append(('remove', kr, pot_rem[kr]))
  for kf in pot_flip.keys():
    pot_actions.append(('flip', kf, pot_flip[kf]))

  return pot_actions


### 2.2 Finishing the algorithm

The search starts from an empty DAG. Then we generate the existing actions until convergence, using the `generate_potential_actions` function above. This returns a list of triplets in the form (action, edge, score difference). We sample these with probabilities proportional to the score and perform the step using the `np.random.choice` function. The algorithm should stop if we already choose a score difference smaller than epsilon and return with `model`!

In [ ]:
def hill_climb(df_data, score, iters, eps):
  model = pgmpy.base.DAG()
  model.add_nodes_from(df_data.columns)
  for i in range(iters):

    # generate potential actions
    pot_actions = generate_potential_actions(model, df_data, score)

    # select action
    scores = [action[2] if action[2] > 0 else 0 for action in pot_actions]
    probs = np.array(scores)
    if(np.all(probs < eps)):
      print(f'Iter {i}: Ended')
      break
    probs /= probs.sum()
    idx = np.random.choice(range(len(pot_actions)), 1, p=probs)[0]
    best_act = pot_actions[idx]

    # excecuting action:
    if (best_act[0] == 'add'):
      print(f'Iter {i}: Added {best_act[1]}')
      u, v = best_act[1]
      model.add_edge(u, v)
    elif(best_act[0] == 'remove'):
      print(f'Iter {i}: Removed {best_act[1]}')
      u, v = best_act[1]
      model.remove_edge(u, v)
    elif(best_act[0] == 'flip'):
      print(f'Iter {i}: Flipped {best_act[1]}')
      u, v = best_act[1]
      model.remove_edge(u,v)
      model.add_edge(v,u)
  return model

### 2.3 Run the algorithm

Choose at least two score functions from the pgmpy [documentation](https://pgmpy.org/structure_estimator/hill.html#structure-score) and try the algorithm. Remember, most of these are discrete variables, so use the discretized `discrete_s_data` dataframe.

In [ ]:
score_fn = pgmpy.estimators.BIC(discrete_s_data).local_score
model = hill_climb(discrete_s_data, score_fn, 100, 0.00001)

Iter 0: Added ('PKC', 'praf')
Iter 1: Added ('plcg', 'PKA')
Iter 2: Added ('plcg', 'praf')
Iter 3: Added ('pakts473', 'p44/42')
Iter 4: Added ('PKC', 'P38')
Iter 5: Added ('PIP2', 'plcg')
Iter 6: Added ('pjnk', 'PKC')
Iter 7: Added ('PKA', 'pmek')
Iter 8: Added ('PIP3', 'plcg')
Iter 9: Added ('plcg', 'pakts473')
Iter 10: Added ('pmek', 'PKC')
Iter 11: Added ('PKA', 'p44/42')
Iter 12: Added ('PIP3', 'PIP2')
Iter 13: Added ('pmek', 'praf')
Iter 14: Added ('pakts473', 'PKA')
Iter 15: Added ('pjnk', 'pakts473')
Iter 16: Added ('plcg', 'pjnk')
Iter 17: Added ('plcg', 'pmek')
Iter 18: Added ('pjnk', 'P38')
Iter 19: Removed ('plcg', 'praf')
Iter 20: Added ('PIP3', 'pjnk')
Iter 21: Removed ('PKC', 'praf')
Iter 22: Added ('pjnk', 'praf')
Iter 23: Ended


In [ ]:
nx_graph_hill_climb = nx.DiGraph(model.edges())
nx.draw(nx_graph_hill_climb, pos=nx.circular_layout(s_graph), with_labels=True)
plt.title('Hill climbing')
plt.show()

How do the tested score functions differ? To what extent can they reproduce the graph? Let's focus on qualitative results for now!

TODO: evaluation

## Exercise 3: Continuous optimization



For continuous optimisation, we take a different approach than before. Here our starting point is the adjacency matrix of the directed graph. Our algorithm of choice is [DAGs with NO TEARS](https://arxiv.org/pdf/1803.01422.pdf), it makes the following assumption: Our data is generated by a linear Structural Equation Model ([SEM](https://en.wikipedia.org/wiki/Structural_equation_modeling)) Therefore, our data matrix is generated by multiplication of itself and the adjacency matrix that generates it:

$X = XW$

### 3.1 The loss

Thus, choosing the appropriate loss function, the first term of the loss is a squared error on the reconstruction:

 $\ell (W; X) := \frac{1}{2n} \Vert X - X W \Vert _F^2$

 Where the definition of Frobenius norm is:

$ \Vert A \Vert _F = \sqrt{ \sum_i \sum_j A_{ij}^2} $


 From this, calculate the gradient of the loss with regards to $W$. After this complete the loss function  `l2_loss_fn`, so that both the loss and the gradient are returned! To do this, we only need the input variable `W` of the adjacency matrix and the data set stored in `self.X`.


In [ ]:
def l2_loss_fn(self, W):
    M = self.X @ W
    R = self.X - M
    loss = ... #TODO
    G_loss = - 1.0 / self.X.shape[0] * self.X.T @ R
    return loss, G_loss

### 3.2 DAG constraint

We can see that the above loss results in an adjacency matrix, but not necessarily a DAG. In order to quantify how "DAG-like" a neighbourhood matrix is, we need to introduce a new function. It is good if this function is also differentiable, so that it can be optimized using a gradient-based method.

Optionally: with a little thought, let's try to figure out together what form we are looking for for this function:

(If you're only interested in the implementation, skip to "The formula"!)

##### Hint #1



The power $W^k$ of the adjacency matrix $W \in \{0,1\}^{d\times d}$ contains the number of $k$ long circles starting from a given vertex in its diagonal

##### Hint #2

The definition of matrix exponential is: $e^X=\sum_{k=0}^{\infty} \frac{1}{k !} X^k$

##### Hint #3

$tr(W^0) = tr(I) = d$

##### The formula



It is important to note that we have considered binary matrices above, but no such restriction has been made for the algorithm. Fortunately, for positive matrices, the formula works in the same way, thus we will use the elementwise multiplication of $W$ by itself: $W \circ W$, so that we get a positive matrix. The final formula for the DAG constraint is:

$h(W) = tr(e^{W \circ W}) - d$

We can see that the smaller this number is, the fewer circles the graph will contain.


#####Continue here
Calculate the gradient of our function $h$ and implement the function `h_fn` as before. Use the function `slin.expm` to compute the matrix exponential.

In [ ]:
def h_fn(self, W):
    d = W.shape[0]
    E = ... # TODO: exponential expression
    h = ... # TODO: as given above
    G_h = E.T * W * 2
    return h, G_h

Hereafter, the method uses an augmented lagrange function and L-BFGS algorithm to minimize the funtions to find $W$ iteratively, see `linear_notears`, this can be done in about 25 lines of code, but is beyond the scope of the lab.

Let's run the algorithm and see the results.

In [ ]:
class NOTEARS():

  def __init__(self, eps=1e-8, rho_max=1e+16, w_threshold=0.3, lambda1=0.1):
    self.eps = eps
    self.rho_max = rho_max
    self.w_threshold = w_threshold
    self.lambda1 = lambda1

  l2_loss = l2_loss_fn
  h = h_fn

  def adj(self, w):
    return (w[:self.d * self.d] - w[self.d * self.d:]).reshape([self.d, self.d])

  def _func(self, w):
    # function to minimize
    W = self.adj(w)

    loss, G_loss = self.l2_loss(W)
    h, G_h = self.h(W)

    obj = loss + 0.5 * self.rho * h * h + self.alpha * h + self.lambda1 * np.absolute(W).sum()
    G_smooth = G_loss + (self.rho * h + self.alpha) * G_h
    g_obj = np.concatenate((G_smooth + self.lambda1, - G_smooth + self.lambda1), axis=None)

    return obj, g_obj

  def linear_notears(self, data, iters):
    self.X = data
    n, self.d = self.X.shape
    w_est, self.rho, self.alpha, h = np.zeros(2 * self.d * self.d), 1.0, 0.0, np.inf
    bnds = [(0, 0) if i == j else (0, None) for _ in range(2) for i in range(self.d) for j in range(self.d)]

    self.X = self.X - np.mean(self.X, axis=0, keepdims=True)
    for _ in range(iters):
      w_new, h_new = None, None
      while self.rho < self.rho_max:
        sol = sopt.minimize(self._func, w_est, method='L-BFGS-B', jac=True, bounds=bnds)
        w_new = sol.x
        h_new, _ = self.h(self.adj(w_new))
        if h_new > 0.25 * h:
          self.rho *= 10
        else:
          break
      w_est, h = w_new, h_new
      self.alpha += self.rho * h
      if h <= self.eps or self.rho >= self.rho_max:
          break
    W_est = self.adj(w_est)
    W_est[np.abs(W_est) < self.w_threshold] = 0
    return W_est

In [ ]:
data = s_data.to_numpy()
nt = NOTEARS()
notears_graph = nt.linear_notears(data, 100)


In [ ]:
nx_graph_notears = nx.from_numpy_array(notears_graph, create_using=nx.DiGraph)
nx_graph_notears = nx.relabel_nodes(nx_graph_notears, {i: name for (i,name) in zip(range(11), s_graph.nodes())})
nx.draw(nx_graph_notears, pos=nx.circular_layout(s_graph), with_labels=True)
plt.title('NOTEARS')
plt.show()

Evaluate our resulting graph qualitatively.

TODO: evaluation

## Exercise 4: Evaluate the results

### 4.1 Defining metrics

For a quick and visual evaluation, we use the Structural Hamming Distance (SHD), which is the sum of missing, falsely discovered and reversed edges. It is defined for three cases: directed, undirected and partially directed graphs.

Supplement the function `directed_shd` based on `undirected_shd`. Note that the direction of the edges matters here!

In [ ]:
def directed_shd(pred_edges, truth_edges):
  # TODO
  pass

def undirected_shd(pred_edges, truth_edges):
  shd = 0
  for (a,b) in truth_edges:
    if (a,b) not in pred_edges and (b,a) not in pred_edges :
      shd += 1
  for (a,b) in pred_edges:
    if (a,b) not in truth_edges and (b,a) not in truth_edges:
      shd += 1
  return shd

def partial_shd(pred_directed, truth_directed, pred_undirected, truth_undirected):
  d = directed_shd(pred_directed, truth_directed)
  u = undirected_shd(pred_undirected, truth_undirected)
  return d + u


### 4.2 Definition of Markov equivalence class

Structure learning typically cannot reconstruct the entire DAG from observed data, only to the extent of the Markov equivalence class. The evaluation therefore requires the comparability of the equivalence classes found. To do this, run the function `create_pdag` function!

The function first creates the skeleton, i.e. the existing edges, which are undirected by default. Then it identifies the colliding V structures (these are definitely directed). Then it directs the possible edges by applying [4 Meek rules](https://arxiv.org/ftp/arxiv/papers/1302/1302.4972.pdf) sequentially.

In [ ]:
def filter_immoralities(immoralities):
  #filtering those that are having edges:
  true_im = []
  for im in immoralities:
    if len(immoralities[im]) > 0:
      true_im.append(immoralities[im][0])
  return true_im

def create_pdag(dag):
  dag = pgmpy.base.DAG(dag)

  # skeleton
  skeleton = list(dag.edges())
  skeleton.extend([(b, a) for (a,b) in skeleton])
  directed = []

  # colliders
  immoralities = filter_immoralities(dag.get_immoralities())

  for (a, b) in immoralities:
    a_children = set(dag.get_children(a))
    b_children = set(dag.get_children(b))
    common_children = list(a_children.intersection(b_children))
    a_edges = [(a, c) for c in common_children]
    b_edges = [(b, c) for c in common_children]
    directed.extend(a_edges)
    directed.extend(b_edges)

  # meek rules
  next_dag = pgmpy.base.DAG()
  next_dag.add_nodes_from(dag.nodes())
  next_dag.add_edges_from(directed)
  previous_dag = None

  while next_dag != previous_dag:
    previous_dag = next_dag
    # 1
    for (a, b) in skeleton:
      if (a, b) in next_dag.edges() or (b, a) in next_dag.edges():
        continue
      if np.all([(c, b) not in skeleton for c in next_dag.get_parents(a)]):
        next_dag.add_edge(a,b)

    # 2
    for (a, b) in skeleton:
      if (a, b) in next_dag.edges() or (b, a) in next_dag.edges():
        continue
      if nx.has_path(next_dag, a, b):
        next_dag.add_edge(a,b)
    # 3
    for (a, b) in skeleton:
      if (a, b) in next_dag.edges() or (b, a) in next_dag.edges():
        continue
      for (i1, i2) in filter_immoralities(next_dag.get_immoralities()):
        if i1 in next_dag.get_parents(b) and i2 in next_dag.get_parents(b) and (a, i1) in skeleton and (a, i2) in skeleton:
          next_dag.add_edge(a,b)
    #4
    for (a, b) in skeleton:
      if (a, b) in next_dag.edges() or (b, a) in next_dag.edges():
        continue
      for pa_b in next_dag.get_parents(b):
        if (pa_b,a) in skeleton:
          for pa_pa_b in next_dag.get_parents(pa_b):
            if (pa_pa_b, a) in skeleton:
              next_dag.add_edge(a,b)
              break

  directed.extend(list(next_dag.edges()))

  undirected = []
  for (a,b) in skeleton:
    if (a,b) not in undirected and (b,a) not in undirected:
      if (a,b) not in directed and (b,a) not in directed:
        undirected.append((a,b))


  pdag = pgmpy.base.PDAG(directed_ebunch=directed,
                         undirected_ebunch=undirected)

  return pdag

### 4.3 Evaluation of graphs

Run the evaluation and evaluate the results and answer the questions.
- Which method best approximates the real graph?
- What guarantees in each method that the resulting graph is DAG?

In [ ]:
true_dag = s_graph
true_pdag = create_pdag(s_graph)
pred_dags = [nx_graph_hill_climb, nx_graph_notears]

# Mut-info graph
mut_info_undirected_shd = undirected_shd(nx_graph_mut_info.edges(), true_dag.edges())
print(f'Hill climb undirected shd: {mut_info_undirected_shd}')

# Hill climb graph
hill_directed_shd = directed_shd(nx_graph_hill_climb.edges(), true_dag.edges())
hill_undirected_shd = undirected_shd(nx_graph_hill_climb.edges(), true_dag.edges())

hill_pdag = create_pdag(nx_graph_hill_climb)
hill_partial_shd = partial_shd(hill_pdag.directed_edges, true_pdag.directed_edges,
                    hill_pdag.undirected_edges, true_pdag.undirected_edges)

print(f'Hill climb directed shd: {hill_directed_shd}, undirected shd: {hill_undirected_shd}, partially directed shd: {hill_partial_shd}')


# Notears graph
notears_directed_shd = directed_shd(nx_graph_notears.edges(), true_dag.edges())
notears_undirected_shd = undirected_shd(nx_graph_notears.edges(), true_dag.edges())

notears_pdag = create_pdag(nx_graph_notears)
notears_partial_shd = partial_shd(notears_pdag.directed_edges, true_pdag.directed_edges,
                    notears_pdag.undirected_edges, true_pdag.undirected_edges)

print(f'NOTEARS directed shd: {notears_directed_shd}, undirected shd: {notears_undirected_shd}, partially directed shd: {notears_partial_shd}')




Hill climb undirected shd: 16
Hill climb directed shd: 13, undirected shd: 17, partially directed shd: 18
NOTEARS directed shd: 13, undirected shd: 22, partially directed shd: 21


In [ ]:
nx.draw_networkx_nodes(true_pdag, pos=nx.circular_layout(s_graph))
nx.draw_networkx_labels(true_pdag, pos=nx.circular_layout(s_graph))
nx.draw_networkx_edges(true_pdag, pos=nx.circular_layout(s_graph), edgelist=true_pdag.directed_edges)
nx.draw_networkx_edges(true_pdag, pos=nx.circular_layout(s_graph), edgelist=true_pdag.undirected_edges, arrows=False)
plt.title('True PDAG')
plt.show()

In [ ]:
nx.draw_networkx_nodes(hill_pdag, pos=nx.circular_layout(s_graph))
nx.draw_networkx_labels(hill_pdag, pos=nx.circular_layout(s_graph))
nx.draw_networkx_edges(hill_pdag, pos=nx.circular_layout(s_graph), edgelist=hill_pdag.directed_edges)
nx.draw_networkx_edges(hill_pdag, pos=nx.circular_layout(s_graph), edgelist=hill_pdag.undirected_edges, arrows=False)
plt.title('Hill Climb PDAG')
plt.show()

In [ ]:
nx.draw_networkx_nodes(notears_pdag, pos=nx.circular_layout(s_graph))
nx.draw_networkx_labels(notears_pdag, pos=nx.circular_layout(s_graph))
nx.draw_networkx_edges(notears_pdag, pos=nx.circular_layout(s_graph), edgelist=notears_pdag.directed_edges)
nx.draw_networkx_edges(notears_pdag, pos=nx.circular_layout(s_graph), edgelist=notears_pdag.undirected_edges, arrows=False)
plt.title('NOTEARS PDAG')
plt.show()

TODO: evaluate results